
# Illustrious XL LoRA Training Notebook

This notebook provisions a full training workflow for Illustrious XL LoRA fine-tuning on Lambda Stack 22.04 (arm64). Work through the sections sequentially: configure the project structure, validate dataset pairs, customize hyperparameters, and finally launch the Accelerate-powered trainer.



## Parameter quick reference

| Section | Parameter | Purpose |
| --- | --- | --- |
| **Run metadata** | `experiment.run_name` | Prefix for checkpoints, previews, and log folders. |
|  | `experiment.output_dir` | Destination directory where model artifacts, logs, and temporary files are written. |
|  | `experiment.seed` | Global random seed applied to Python, NumPy, and PyTorch for reproducibility. |
| **Model preparation** | `model.pretrained_model_name_or_path` | Hugging Face repository ID or local path for the base Illustrious/SDXL checkpoint (e.g., `OnomaAIResearch/Illustrious-xl-early-release-v0`). |
| **Optimizer setup** | `optimizer.mixed_precision` | Mixed precision mode (`"bf16"`, `"fp16"`, or `"no"`) forwarded to `Accelerator`. |
|  | `optimizer.use_8bit_adam` | Toggle for bitsandbytes AdamW optimizer to reduce VRAM footprint. |
| **Training hyperparameters** | `training.train_batch_size` | Mini-batch size per forward pass (before gradient accumulation). |
|  | `training.gradient_accumulation_steps` | Number of accumulation steps that together form one optimizer update. |
|  | `training.learning_rate` | Base learning rate for AdamW. |
|  | `training.lr_scheduler` | Scheduler strategy (`constant`, `cosine`, or `cosine_with_restarts`). |
|  | `training.lr_warmup_steps` | Scheduler warmup steps prior to LR decay. |
|  | `training.max_train_epochs` | Number of epochs to iterate across the dataset. |
|  | `training.checkpointing_steps` | Interval (in optimizer steps) for saving LoRA checkpoints. |
|  | `training.dataloader_num_workers` | Worker count for the PyTorch `DataLoader`. |
| **LoRA configuration** | `lora.network_rank` | Rank of the LoRA adapters for attention processors. |
|  | `lora.network_alpha` | Scaling factor applied to LoRA weights. |
| **Dataset setup** | `dataset.root` | Base path containing user-uploaded `.jpg/.txt` pairs. |
|  | `dataset.train_folder` | Subdirectory relative to `root` that holds training pairs. |
|  | `dataset.caption_extension` | Required caption suffix for each image (defaults to `.txt`). |
|  | `dataset.image_extensions` | Allowed image extensions for ingestion. |
|  | `dataset.max_train_images` | Cap on how many examples to include (defaults to 100). |
|  | `dataset.resolution` | Square resolution for preprocessing (Illustrious XL defaults to `1024`). |
|  | `dataset.shuffle_before_validation` | Shuffle toggle before selecting validation prompts. |
| **Validation preview** | `validation.prompt` / `validation.negative_prompt` | Prompts used for periodic preview renders. |
|  | `validation.num_images` | Number of validation renders to generate with the trained adapter. |
| **Concept** | `concept.trigger_token` | Token inserted into captions/prompts to activate the concept. |
|  | `concept.prior_loss_weight` | Weight for the prior preservation loss term. |



## Project layout templates

The notebook recognizes the default multi-project structure and an extended alternative:

```
Loras/
├─ _logs/
├─ project_name/
│  ├─ configs/
│  │  ├─ dataset_config.toml
│  │  └─ training_config.toml
│  ├─ dataset/
│  │  └─ train/
│  │     ├─ *.jpg
│  │     └─ *.txt
│  └─ output/
│     ├─ checkpoints/
│     ├─ previews/
│     └─ tensorboard/
└─ shared_assets/
   ├─ reference/
   └─ validation/
```

An extended variant for multi-concept experiments keeps base assets decoupled:

```
Loras_ext/
├─ cache/
├─ concepts/
│  └─ <concept_id>/
│     ├─ dataset/
│     ├─ configs/
│     └─ output/
├─ shared/
│  ├─ embeddings/
│  ├─ validation_prompts/
│  └─ wildcards/
└─ utilities/
   └─ scripts/
```

Update `project_root` in the next cell if you adopt the extended layout.


In [ ]:

from pathlib import Path
from pprint import pprint

project_root = Path.cwd()
project_root.mkdir(parents=True, exist_ok=True)

layouts = {
    "default": project_root / "Loras",
    "extended": project_root / "Loras_ext",
}

for name, layout in layouts.items():
    for sub in [
        layout / "_logs",
        layout / "shared_assets",
        layout / "shared_assets/reference",
        layout / "project_name/configs",
        layout / "project_name/dataset/train",
        layout / "project_name/output/checkpoints",
        layout / "project_name/output/previews",
        layout / "project_name/output/tensorboard",
    ]:
        sub.mkdir(parents=True, exist_ok=True)
    print(f"Prepared '{name}' layout at {layout}")

pprint({name: str(path) for name, path in layouts.items()})



## Dependency bootstrap

Run the following cell once per environment to ensure all notebook dependencies are available (arm64 wheels are published for each package listed). Comment out the line if your environment already satisfies the requirement.


In [ ]:

%pip install --upgrade --quiet accelerate==1.3.0 diffusers==0.31.0 transformers==4.46.3 datasets==3.1.0 torchvision==0.20.1 tomlkit==0.13.2 peft==0.13.2 bitsandbytes==0.43.3 safetensors==0.4.5


In [ ]:

import importlib

required_packages = [
    "accelerate",
    "datasets",
    "diffusers",
    "PIL",
    "torch",
    "torchvision",
    "transformers",
    "peft",
    "safetensors",
]
missing = []
for pkg in required_packages:
    pkg_name = pkg.lower().split(":")[0]
    if importlib.util.find_spec(pkg_name) is None:
        missing.append(pkg)
if missing:
    raise ImportError(f"Missing packages detected: {missing}. Please install them via %pip before proceeding.")

print("Environment check passed. All required packages are importable.")


In [ ]:

import math
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

import torch
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import ProjectConfiguration, set_seed
from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionXLPipeline
from diffusers.models.attention_processor import (
    AttnProcessor,
    AttnProcessor2_0,
    LoRAAttnProcessor,
    LoRAAttnProcessor2_0,
)
from PIL import Image
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from transformers import CLIPTokenizer, CLIPTextModel, CLIPTextModelWithProjection
import tomllib
from peft import LoraConfig as PeftLoraConfig, get_peft_model

logger = get_logger(__name__)


In [ ]:

@dataclass
class ExperimentConfig:
    run_name: str
    output_dir: Path
    seed: int = 42

@dataclass
class ModelConfig:
    pretrained_model_name_or_path: str

@dataclass
class OptimizerConfig:
    mixed_precision: str = "bf16"
    use_8bit_adam: bool = True

@dataclass
class TrainingConfig:
    train_batch_size: int = 1
    gradient_accumulation_steps: int = 1
    learning_rate: float = 1e-4
    lr_scheduler: str = "constant"
    lr_warmup_steps: int = 0
    max_train_epochs: int = 1
    checkpointing_steps: int = 500
    dataloader_num_workers: int = 4
    use_sdpa: bool = True
    gradient_checkpointing: bool = True
    allow_tf32: bool = True

@dataclass
class LoRAConfig:
    network_rank: int = 8
    network_alpha: int = 16

@dataclass
class DatasetConfig:
    root: Path
    train_folder: str
    caption_extension: str = ".txt"
    image_extensions: Tuple[str, ...] = (".jpg", ".png")
    max_train_images: int = 100
    resolution: int = 1024
    shuffle_before_validation: bool = True

@dataclass
class ValidationConfig:
    prompt: str
    negative_prompt: str = ""
    num_images: int = 4

@dataclass
class ConceptConfig:
    trigger_token: str
    prior_loss_weight: float = 0.1

@dataclass
class NotebookConfig:
    experiment: ExperimentConfig
    model: ModelConfig
    optimizer: OptimizerConfig
    training: TrainingConfig
    lora: LoRAConfig
    dataset: DatasetConfig
    validation: ValidationConfig
    concept: ConceptConfig


def _load_toml(path: Path) -> Dict:
    with open(path, "rb") as fh:
        return tomllib.load(fh)


def load_training_config(path: Path) -> NotebookConfig:
    data = _load_toml(path)
    experiment = ExperimentConfig(
        **data["experiment"],
        output_dir=Path(data["experiment"]["output_dir"]),
    )
    model = ModelConfig(**data["model"])
    optimizer = OptimizerConfig(**data.get("optimizer", {}))
    training = TrainingConfig(**data.get("training", {}))
    lora = LoRAConfig(**data.get("lora", {}))
    dataset_section = data.get("dataset")
    validation_section = data.get("validation")
    concept_section = data.get("concept")
    if dataset_section is None or validation_section is None or concept_section is None:
        raise ValueError("Training configuration file is missing dataset/validation/concept sections.")
    dataset = DatasetConfig(
        root=Path(dataset_section["root"]),
        train_folder=dataset_section["train_folder"],
        caption_extension=dataset_section.get("caption_extension", ".txt"),
        image_extensions=tuple(dataset_section.get("image_extensions", [".jpg", ".png"])),
        max_train_images=dataset_section.get("max_train_images", 100),
        resolution=dataset_section.get("resolution", 1024),
        shuffle_before_validation=dataset_section.get("shuffle_before_validation", True),
    )
    validation = ValidationConfig(**validation_section)
    concept = ConceptConfig(**concept_section)
    return NotebookConfig(
        experiment=experiment,
        model=model,
        optimizer=optimizer,
        training=training,
        lora=lora,
        dataset=dataset,
        validation=validation,
        concept=concept,
    )


In [ ]:

@dataclass
class DatasetEntry:
    image_path: Path
    caption_path: Path
    caption: str


def scan_dataset(cfg: DatasetConfig) -> List[DatasetEntry]:
    dataset_root = (cfg.root if cfg.root.is_absolute() else Path.cwd() / cfg.root).resolve()
    train_dir = (dataset_root / cfg.train_folder).resolve()
    if not train_dir.exists():
        raise FileNotFoundError(f"Training folder {train_dir} was not found.")

    entries: List[DatasetEntry] = []
    supported = {ext.lower() for ext in cfg.image_extensions}
    for image_path in sorted(train_dir.glob("**/*")):
        if image_path.suffix.lower() not in supported:
            continue
        caption_path = image_path.with_suffix(cfg.caption_extension)
        if not caption_path.exists():
            logger.warning("Missing caption for %s", image_path.name)
            continue
        caption = caption_path.read_text(encoding="utf-8").strip()
        if not caption:
            logger.warning("Empty caption file detected at %s", caption_path)
            continue
        entries.append(DatasetEntry(image_path=image_path, caption_path=caption_path, caption=caption))
        if len(entries) >= cfg.max_train_images:
            break

    if not entries:
        raise RuntimeError("No valid image/caption pairs were discovered. Please verify your dataset.")

    logger.info("Discovered %d training pairs (capped at %d).", len(entries), cfg.max_train_images)
    return entries


In [ ]:

class IllustriousDataset(Dataset):
    def __init__(self, entries: List[DatasetEntry], resolution: int):
        self.entries = entries
        self.resolution = resolution
        self.transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=InterpolationMode.BILINEAR, antialias=True),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self) -> int:
        return len(self.entries)

    def __getitem__(self, index: int) -> Dict:
        entry = self.entries[index]
        with Image.open(entry.image_path) as img:
            image = img.convert("RGB")
            original_size = (image.height, image.width)
            pixel_values = self.transform(image)
        return {
            "pixel_values": pixel_values,
            "caption": entry.caption,
            "original_size": original_size,
            "crop_coords": (0, 0),
        }


def collate_examples(examples: List[Dict]) -> Dict:
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    captions = [example["caption"] for example in examples]
    original_sizes = torch.tensor([example["original_size"] for example in examples], dtype=torch.long)
    crop_coords = torch.tensor([example["crop_coords"] for example in examples], dtype=torch.long)
    return {
        "pixel_values": pixel_values,
        "captions": captions,
        "original_sizes": original_sizes,
        "crop_coords": crop_coords,
    }



## Load configuration files

Update the paths below if you created custom TOML files. The defaults reference the starter examples inside `project_name/configs/` and target a 100-image dataset.


In [ ]:

training_config_path = Path("project_name/configs/training_config.example.toml")
dataset_config_path = Path("project_name/configs/dataset_config.example.toml")

notebook_config = load_training_config(training_config_path)
dataset_entries = scan_dataset(notebook_config.dataset)

display({
    "experiment": notebook_config.experiment,
    "model": notebook_config.model,
    "optimizer": notebook_config.optimizer,
    "training": notebook_config.training,
    "lora": notebook_config.lora,
    "dataset_size": len(dataset_entries),
})


In [ ]:

# Preview a handful of captions to confirm ingestion order.
for sample in dataset_entries[:5]:
    print(sample.image_path.name)
    print("  caption:", sample.caption[:160])



## Optional: Hugging Face authentication

Authenticate against the Hugging Face Hub if the selected Illustrious checkpoint requires gated access. Skip this cell if you already stored your token in `huggingface-cli login`.


In [ ]:

from huggingface_hub import login

use_hf_login = False  # Set to True to enable interactive login.
if use_hf_login:
    login()



## Trainer utilities

The following helpers construct the SDXL components, attach LoRA adapters, initialize Accelerate, and implement the end-to-end training routine with checkpointing and validation previews.


In [ ]:

from itertools import chain
from typing import Iterable

from diffusers.loaders import LoraLoaderMixin
from torch.optim import AdamW
from transformers import get_scheduler


def determine_dtype(mixed_precision: str) -> torch.dtype:
    mixed_precision = (mixed_precision or "no").lower()
    if mixed_precision == "fp16":
        return torch.float16
    if mixed_precision == "bf16":
        return torch.bfloat16
    return torch.float32


def create_accelerator(config: NotebookConfig) -> Accelerator:
    project_dir = Path(config.experiment.output_dir) / "accelerate"
    project_dir.mkdir(parents=True, exist_ok=True)
    logging_dir = Path(config.experiment.output_dir) / "logs"
    logging_dir.mkdir(parents=True, exist_ok=True)
    project_config = ProjectConfiguration(project_dir=str(project_dir), logging_dir=str(logging_dir))
    accelerator = Accelerator(
        mixed_precision=None if config.optimizer.mixed_precision == "no" else config.optimizer.mixed_precision,
        gradient_accumulation_steps=config.training.gradient_accumulation_steps,
        log_with="tensorboard",
        project_config=project_config,
    )
    if accelerator.is_main_process:
        (Path(config.experiment.output_dir) / "output" / "tensorboard").mkdir(parents=True, exist_ok=True)
    return accelerator


def setup_pipeline(config: NotebookConfig, dtype: torch.dtype) -> StableDiffusionXLPipeline:
    pipeline = StableDiffusionXLPipeline.from_pretrained(
        config.model.pretrained_model_name_or_path,
        torch_dtype=dtype,
        variant="fp16" if dtype == torch.float16 else None,
        use_safetensors=True,
    )
    pipeline.to("cpu")
    if config.training.use_sdpa:
        pipeline.enable_attention_slicing(None)
        pipeline.unet.set_default_attn_processor()
    pipeline.vae.requires_grad_(False)
    pipeline.vae.eval()
    return pipeline


def patch_unet_with_lora(unet, network_rank: int, network_alpha: int):
    attn_procs = {}
    for name, processor in unet.attn_processors.items():
        if isinstance(processor, (LoRAAttnProcessor, LoRAAttnProcessor2_0)):
            attn_procs[name] = processor
            continue
        if isinstance(processor, AttnProcessor2_0):
            hidden_size = processor.hidden_size
            cross_attention_dim = processor.cross_attention_dim
            attn_procs[name] = LoRAAttnProcessor2_0(
                hidden_size=hidden_size,
                cross_attention_dim=cross_attention_dim,
                rank=network_rank,
                network_alpha=network_alpha,
            )
        elif isinstance(processor, AttnProcessor):
            hidden_size = processor.hidden_size
            cross_attention_dim = processor.cross_attention_dim
            attn_procs[name] = LoRAAttnProcessor(
                hidden_size=hidden_size,
                cross_attention_dim=cross_attention_dim,
                rank=network_rank,
                network_alpha=network_alpha,
            )
        else:
            attn_procs[name] = processor
    unet.set_attn_processor(attn_procs)
    trainable_params = []
    for module in unet.attn_processors.values():
        if hasattr(module, "parameters"):
            for param in module.parameters():
                param.requires_grad_(True)
                trainable_params.append(param)
    return trainable_params


def patch_text_encoders_with_lora(
    text_encoder: CLIPTextModel,
    text_encoder_2: CLIPTextModelWithProjection,
    network_rank: int,
    network_alpha: int,
):
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"]
    config = PeftLoraConfig(
        r=network_rank,
        lora_alpha=network_alpha,
        target_modules=target_modules,
        lora_dropout=0.0,
        bias="none",
        task_type="SEQ_CLS",
    )
    text_encoder_lora = get_peft_model(text_encoder, config)
    text_encoder_2_lora = get_peft_model(text_encoder_2, config)
    return text_encoder_lora, text_encoder_2_lora


def create_dataloader(entries: List[DatasetEntry], config: NotebookConfig) -> DataLoader:
    dataset = IllustriousDataset(entries, config.dataset.resolution)
    return DataLoader(
        dataset,
        batch_size=config.training.train_batch_size,
        shuffle=True,
        num_workers=config.training.dataloader_num_workers,
        pin_memory=True,
        collate_fn=collate_examples,
    )


def prepare_optimizer(parameters: Iterable[torch.nn.Parameter], config: NotebookConfig):
    betas = (0.9, 0.999)
    if config.optimizer.use_8bit_adam:
        try:
            import bitsandbytes as bnb

            optimizer = bnb.optim.AdamW8bit(parameters, lr=config.training.learning_rate, betas=betas, weight_decay=1e-2)
            return optimizer
        except ImportError:
            logger.warning("bitsandbytes is not installed; falling back to standard AdamW.")
    return AdamW(parameters, lr=config.training.learning_rate, betas=betas, weight_decay=1e-2)


def save_lora_checkpoint(accelerator: Accelerator, pipeline: StableDiffusionXLPipeline, config: NotebookConfig, output_subdir: str):
    save_dir = Path(config.experiment.output_dir) / "output" / "checkpoints" / output_subdir
    save_dir.mkdir(parents=True, exist_ok=True)
    pipeline.save_lora_weights(
        save_directory=str(save_dir),
        unet=accelerator.unwrap_model(pipeline.unet),
        text_encoder=accelerator.unwrap_model(pipeline.text_encoder),
        text_encoder_2=accelerator.unwrap_model(pipeline.text_encoder_2),
        safe_serialization=True,
    )
    accelerator.print(f"Saved LoRA checkpoint to {save_dir}")


def generate_validation_previews(accelerator: Accelerator, pipeline: StableDiffusionXLPipeline, config: NotebookConfig, step_identifier: str):
    output_dir = Path(config.experiment.output_dir) / "output" / "previews"
    output_dir.mkdir(parents=True, exist_ok=True)
    generator = torch.Generator(device=accelerator.device).manual_seed(config.experiment.seed)
    images = pipeline(
        prompt=[config.validation.prompt] * config.validation.num_images,
        negative_prompt=[config.validation.negative_prompt] * config.validation.num_images,
        num_inference_steps=30,
        guidance_scale=5.0,
        generator=generator,
    ).images
    for idx, image in enumerate(images):
        preview_path = output_dir / f"preview_step-{step_identifier}_{idx:02d}.png"
        image.save(preview_path)
        accelerator.print(f"Saved preview to {preview_path}")


In [ ]:

def train_lora(
    config: NotebookConfig,
    entries: List[DatasetEntry],
    resume_from_checkpoint: Optional[str] = None,
    run_validation: bool = True,
):
    accelerator = create_accelerator(config)
    dtype = determine_dtype(config.optimizer.mixed_precision)
    set_seed(config.experiment.seed)

    pipeline = setup_pipeline(config, dtype)

    dataloader = create_dataloader(entries, config)

    text_encoder = pipeline.text_encoder
    text_encoder_2 = pipeline.text_encoder_2
    tokenizer_one: CLIPTokenizer = pipeline.tokenizer
    tokenizer_two: CLIPTokenizer = pipeline.tokenizer_2
    unet = pipeline.unet
    vae: AutoencoderKL = pipeline.vae
    noise_scheduler = DDPMScheduler.from_config(pipeline.scheduler.config)

    if config.training.gradient_checkpointing:
        unet.enable_gradient_checkpointing()
        text_encoder.gradient_checkpointing_enable()
        text_encoder_2.gradient_checkpointing_enable()

    unet.requires_grad_(False)
    text_encoder.requires_grad_(False)
    text_encoder_2.requires_grad_(False)

    unet_lora_params = patch_unet_with_lora(unet, config.lora.network_rank, config.lora.network_alpha)
    text_encoder_lora, text_encoder_2_lora = patch_text_encoders_with_lora(
        text_encoder,
        text_encoder_2,
        config.lora.network_rank,
        config.lora.network_alpha,
    )

    trainable_params = list(chain(unet_lora_params, text_encoder_lora.parameters(), text_encoder_2_lora.parameters()))
    trainable_params = [param for param in trainable_params if param.requires_grad]

    optimizer = prepare_optimizer(trainable_params, config)

    if config.optimizer.mixed_precision == "no":
        accelerator.native_amp = False

    num_update_steps_per_epoch = max(1, math.ceil(len(dataloader) / config.training.gradient_accumulation_steps))
    total_train_steps = num_update_steps_per_epoch * config.training.max_train_epochs
    lr_scheduler = get_scheduler(
        name=config.training.lr_scheduler,
        optimizer=optimizer,
        num_warmup_steps=config.training.lr_warmup_steps,
        num_training_steps=total_train_steps,
    )

    (
        text_encoder_lora,
        text_encoder_2_lora,
        unet,
        optimizer,
        dataloader,
        lr_scheduler,
    ) = accelerator.prepare(text_encoder_lora, text_encoder_2_lora, unet, optimizer, dataloader, lr_scheduler)

    weight_dtype = dtype
    vae.to(accelerator.device, dtype=torch.float32)
    noise_scheduler.set_timesteps(noise_scheduler.config.num_train_timesteps)

    if accelerator.is_main_process:
        accelerator.init_trackers(config.experiment.run_name, config={"lr": config.training.learning_rate})

    global_step = 0
    for epoch in range(config.training.max_train_epochs):
        unet.train()
        text_encoder_lora.train()
        text_encoder_2_lora.train()
        for step, batch in enumerate(dataloader):
            with accelerator.accumulate(unet):
                pixel_values = batch["pixel_values"].to(accelerator.device, dtype=weight_dtype)
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                timesteps = torch.randint(
                    0,
                    noise_scheduler.config.num_train_timesteps,
                    (bsz,),
                    device=latents.device,
                )
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                prompts = batch["captions"]
                text_inputs_one = tokenizer_one(
                    prompts,
                    padding="max_length",
                    max_length=tokenizer_one.model_max_length,
                    truncation=True,
                    return_tensors="pt",
                )
                text_inputs_two = tokenizer_two(
                    prompts,
                    padding="max_length",
                    max_length=tokenizer_two.model_max_length,
                    truncation=True,
                    return_tensors="pt",
                )
                text_inputs_one = {k: v.to(accelerator.device) for k, v in text_inputs_one.items()}
                text_inputs_two = {k: v.to(accelerator.device) for k, v in text_inputs_two.items()}

                text_encoder_outputs = text_encoder_lora(**text_inputs_one, output_hidden_states=True)
                text_encoder_outputs_2 = text_encoder_2_lora(**text_inputs_two, output_hidden_states=True)

                prompt_embeds = text_encoder_outputs.hidden_states[-2]
                prompt_embeds_2 = text_encoder_outputs_2.hidden_states[-2]
                prompt_embeds = torch.cat([prompt_embeds, prompt_embeds_2], dim=-1)
                pooled_prompt_embeds = text_encoder_outputs_2.pooler_output

                original_sizes = batch["original_sizes"].to(accelerator.device)
                crop_coords = batch["crop_coords"].to(accelerator.device)
                target_size = (config.dataset.resolution, config.dataset.resolution)
                add_time_ids = pipeline._get_add_time_ids(
                    original_sizes,
                    crop_coords,
                    target_size,
                    accelerator.device,
                )

                model_pred = unet(
                    noisy_latents,
                    timesteps,
                    encoder_hidden_states=prompt_embeds,
                    added_cond_kwargs={
                        "text_embeds": pooled_prompt_embeds,
                        "time_ids": add_time_ids,
                    },
                ).sample

                target = noise
                loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")
                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(trainable_params, max_norm=1.0)

                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            if accelerator.sync_gradients:
                global_step += 1
                if global_step % config.training.checkpointing_steps == 0:
                    save_lora_checkpoint(accelerator, pipeline, config, f"step-{global_step}")
                    if run_validation and accelerator.is_main_process:
                        generate_validation_previews(accelerator, pipeline, config, f"{global_step:06d}")

            logs = {"loss": loss.detach().item(), "lr": lr_scheduler.get_last_lr()[0]}
            accelerator.log(logs, step=global_step)

            if global_step >= total_train_steps:
                break
        if global_step >= total_train_steps:
            break

    accelerator.wait_for_everyone()
    save_lora_checkpoint(accelerator, pipeline, config, "final")
    if run_validation and accelerator.is_main_process:
        generate_validation_previews(accelerator, pipeline, config, "final")
    accelerator.end_training()
    return {
        "global_step": global_step,
        "epochs": config.training.max_train_epochs,
        "total_train_steps": total_train_steps,
    }



## Launch training

Set `enable_training` to `True` after you review the configuration summary. Training creates checkpoints under `project_name/output/checkpoints/` and previews in `project_name/output/previews/`.


In [ ]:

enable_training = False  # Flip to True to start training
run_validation = True

if enable_training:
    metrics = train_lora(
        notebook_config,
        dataset_entries,
        resume_from_checkpoint=None,
        run_validation=run_validation,
    )
    display(metrics)
else:
    print("Training is disabled. Set enable_training = True to begin.")



## Post-training: Load adapter for inference

Once training finishes, run this cell to attach the latest LoRA weights and test prompt rendering inside the notebook or any compatible downstream pipeline.


In [ ]:

from diffusers import DiffusionPipeline

lora_checkpoint_dir = Path(notebook_config.experiment.output_dir) / "output" / "checkpoints" / "final"
base_pipeline = DiffusionPipeline.from_pretrained(
    notebook_config.model.pretrained_model_name_or_path,
    torch_dtype=determine_dtype(notebook_config.optimizer.mixed_precision),
)
base_pipeline.load_lora_weights(lora_checkpoint_dir)
base_pipeline.to("cuda" if torch.cuda.is_available() else "cpu")

prompt = notebook_config.validation.prompt
negative_prompt = notebook_config.validation.negative_prompt

with torch.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", dtype=determine_dtype(notebook_config.optimizer.mixed_precision)):
    result = base_pipeline(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=30, guidance_scale=5.0)

for idx, image in enumerate(result.images):
    preview_path = Path(notebook_config.experiment.output_dir) / "output" / "previews" / f"inference_{idx:02d}.png"
    image.save(preview_path)
    display(image)
    print(f"Saved inference preview to {preview_path}")



## Future extensions

Use the following placeholders to integrate advanced features, such as prior preservation data loaders, DreamBooth-specific augmentations, SDPA toggles, or ONNX export. They are intentionally empty so you can stage experiments without modifying the core training flow.


In [ ]:

# Before-training feature staging cell
# Example: add custom dataset filtering, caption rewrites, or statistics here.


In [ ]:

# After-training feature staging cell
# Example: implement ONNX export, merge adapters, or quantization experiments here.
